# Util embedding experiments
Flow: load stimuli -> get embeddings -> fit a direction per subset -> compare directions.

In [1]:
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

HERE = Path.cwd()
sys.path.insert(0, str(HERE))

import datasets
import embed

/home/matthew/Projects/2025_moral_feature_modeling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load stimuli
One (label, texts, y, kind) tuple per subset we fit a direction for.

In [2]:
SUBSETS = []  # (label, texts, y, kind)   kind: "cont" or "bin"
# Small datasets first (fast feedback); large ETHICS train splits last.

# franken
exp1, exp2 = datasets.load_franken()
SUBSETS.append(("franken-valence", exp1["target"].tolist(), exp1["avg_likert_rating"].astype(float).values, "cont"))
SUBSETS.append(("franken-good_vs_harm", exp1["target"].tolist(), (exp1["type"] == "good").astype(float).values, "bin"))
SUBSETS.append(("franken-severe_vs_mild", exp1["target"].tolist(), (exp1["strength"] == "severe").astype(float).values, "bin"))
SUBSETS.append(("franken-permissibility", exp2["text"].tolist(), exp2["avg_permissibility_rating"].astype(float).values, "cont"))
SUBSETS.append(("franken-intention", exp2["text"].tolist(), exp2["avg_intention_rating"].astype(float).values, "cont"))

# nie / MoCa
nie = datasets.load_nie()
SUBSETS.append(("nie-acceptability", nie["text"].tolist(), nie["p_yes"].astype(float).values, "cont"))
for col, pos in [("causal_role", "Means"), ("personal_force", "Personal"),
                 ("evitability", "Inevitable"), ("beneficiary", "Other-beneficial")]:
    sub = nie.dropna(subset=[col])
    if sub[col].nunique() > 1:
        SUBSETS.append((f"nie-{col}", sub["text"].tolist(), (sub[col] == pos).astype(float).values, "bin"))

# AITA
aita = datasets.load_aita()
SUBSETS.append(("AITA-utility", aita["outcome"].tolist(), aita["utility"].astype(float).values, "cont"))

# Holmes-Rahe
hr = datasets.load_holmes_rahe()
SUBSETS.append(("Holmes-Rahe", hr["event"].tolist(), hr["lcu"].astype(float).values, "cont"))

# GBD
gbd = datasets.load_gbd()
SUBSETS.append(("GBD", gbd["lay_description"].tolist(), gbd["weight"].astype(float).values, "cont"))

# ETHICS (large train splits - slowest to embed)
ethics = datasets.load_ethics()
a, b = ethics["utilitarianism"]["train"]["more_pleasant"], ethics["utilitarianism"]["train"]["less_pleasant"]
SUBSETS.append(("ETHICS-util", list(a) + list(b), np.r_[np.ones(len(a)), np.zeros(len(b))], "bin"))

cm = ethics["commonsense"]["train"]
cm_short = cm[cm["is_short"]]
SUBSETS.append(("ETHICS-cm", cm_short["input"].tolist(), cm_short["label"].astype(float).values, "bin"))

deon = ethics["deontology"]["train"]
deon_text = (deon["scenario"] + " " + deon["excuse"]).tolist()
SUBSETS.append(("ETHICS-deon", deon_text, deon["label"].astype(float).values, "bin"))

for label, texts, y, kind in SUBSETS:
    print(f"{label:24s} n={len(texts):5d} kind={kind}")

franken-valence          n=   80 kind=cont
franken-good_vs_harm     n=   80 kind=bin
franken-severe_vs_mild   n=   80 kind=bin
franken-permissibility   n=   80 kind=cont
franken-intention        n=   80 kind=cont
nie-acceptability        n=   44 kind=cont
nie-causal_role          n=   35 kind=bin
nie-personal_force       n=   35 kind=bin
nie-evitability          n=   35 kind=bin
nie-beneficiary          n=   35 kind=bin
AITA-utility             n=   59 kind=cont
Holmes-Rahe              n=   43 kind=cont
GBD                      n=  203 kind=cont
ETHICS-util              n=27476 kind=bin
ETHICS-cm                n= 6661 kind=bin
ETHICS-deon              n=18164 kind=bin


## 2. Get embeddings
Switch `MODEL` to any name in `embed.OPENAI_MODELS` or `embed.QWEN_MODELS` to change model.

In [3]:
import os
MODEL = "Qwen/Qwen3-Embedding-4B"  # or any of embed.OPENAI_MODELS / embed.QWEN_MODELS
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")  # only needed if MODEL is an OpenAI model

embeddings = {}
for label, texts, y, kind in SUBSETS:
    emb = embed.embed_dataset(texts, dataset=label, model=MODEL, api_key=OPENAI_API_KEY)
    X = np.array([emb[str(t)] for t in texts])
    embeddings[label] = X
    print(f"{label:24s} {X.shape}")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 506.70it/s]


[Qwen/Qwen3-Embedding-4B] embedding 80 texts for 'franken-valence'
[Qwen/Qwen3-Embedding-4B] done, cache=80
franken-valence          (80, 2560)
[Qwen/Qwen3-Embedding-4B] embedding 80 texts for 'franken-good_vs_harm'
[Qwen/Qwen3-Embedding-4B] done, cache=80
franken-good_vs_harm     (80, 2560)
[Qwen/Qwen3-Embedding-4B] embedding 80 texts for 'franken-severe_vs_mild'
[Qwen/Qwen3-Embedding-4B] done, cache=80
franken-severe_vs_mild   (80, 2560)
[Qwen/Qwen3-Embedding-4B] embedding 80 texts for 'franken-permissibility'
[Qwen/Qwen3-Embedding-4B] done, cache=80
franken-permissibility   (80, 2560)
[Qwen/Qwen3-Embedding-4B] embedding 80 texts for 'franken-intention'
[Qwen/Qwen3-Embedding-4B] done, cache=80
franken-intention        (80, 2560)
[Qwen/Qwen3-Embedding-4B] embedding 44 texts for 'nie-acceptability'
[Qwen/Qwen3-Embedding-4B] done, cache=42
nie-acceptability        (44, 2560)
[Qwen/Qwen3-Embedding-4B] embedding 35 texts for 'nie-causal_role'
[Qwen/Qwen3-Embedding-4B] done, cache=33
nie-c

## 3. Fit directions
Two simple methods to compare: ridge regression of the embeddings on `y`, and a plain
mean-difference between the high and low halves (split at the median for continuous `y`).

In [4]:
def ridge_dir(X, y, alpha=50.0):
    Xc = X - X.mean(0)
    w = np.linalg.solve(Xc.T @ Xc + alpha * np.eye(Xc.shape[1]), Xc.T @ (y - y.mean()))
    return w / (np.linalg.norm(w) + 1e-9)

def meandiff_dir(X, y):
    classes = np.unique(y)
    if len(classes) == 2:            # binary: split by class membership, not by a threshold
        hi, lo = X[y == classes[1]].mean(0), X[y == classes[0]].mean(0)
    else:                            # continuous: split at the median
        thresh = np.median(y)
        hi, lo = X[y >= thresh].mean(0), X[y < thresh].mean(0)
    d = hi - lo
    return d / (np.linalg.norm(d) + 1e-9)

METHOD = "ridge"  # or "meandiff"

directions = {}
for label, texts, y, kind in SUBSETS:
    X = embeddings[label]
    d = ridge_dir(X, y.astype(float)) if METHOD == "ridge" else meandiff_dir(X, y.astype(float))
    directions[label] = d

## 4. Correlation between y and projection
For each subset, project its own embeddings onto its fitted direction and correlate
with the true `y`. In-sample (not held-out) - a quick sanity check that the direction
actually points the way it should, not a decodability estimate.

In [5]:
labels_c, rs = [], []
for label, texts, y, kind in SUBSETS:
    proj = embeddings[label] @ directions[label]
    rs.append(np.corrcoef(proj, y.astype(float))[0, 1])
    labels_c.append(label)

fig = go.Figure(go.Bar(x=labels_c, y=rs, text=rs, texttemplate="%{text:.2f}", textposition="outside"))
fig.update_layout(title="In-sample correlation between y and projection onto fitted direction",
                   yaxis_title="Pearson r", xaxis_tickangle=-60, width=900, height=500)
fig.show()

## 5. Cosine similarity between directions

In [6]:
labels = list(directions.keys())
D = np.array([directions[l] for l in labels])
C = D @ D.T

fig = px.imshow(C, x=labels, y=labels, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Direction x direction cosine similarity", width=900, height=850)
fig.show()